In [27]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, shape
from shapely import wkt
import altair as alt
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import shutil
import subprocess
import os
import folium
import folium.plugins
import datetime
alt.data_transformers.disable_max_rows()
import warnings
import os, glob, time
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")



os.environ["JAVA_HOME"] = subprocess.run(
    ["/usr/libexec/java_home", "-v", "21"],
    capture_output=True, text=True, check=True
).stdout.strip()


from r5py.util.config import Config
import r5py

In [28]:
neigh = pd.read_csv("Data/Neigh/neigh.csv")

neigh["geometry"] = neigh["geom"].apply(wkt.loads)

neigh = gpd.GeoDataFrame(
    neigh,
    geometry="geometry",
    crs="EPSG:4326"
)
neigh = neigh[["name", "geometry"]]
neigh["geometry"] = neigh.buffer(0)


neigh_layer = alt.Chart(neigh).mark_geoshape(
    stroke="black",
    fill="lightblue"
).properties(
    width=600,
    height=600
    
).encode(tooltip=['name'])

In [29]:
# bus_sol = pd.read_csv('Nodes/N-Bus-Sol.csv')
# fgc_sol = pd.read_csv('Nodes/N-FGC-Sol.csv')
# metro_sol = pd.read_csv('Nodes/N-Metro-Sol.csv')
# tram_sol = pd.read_csv('Nodes/N-Tram-Sol.csv')

# all_stops = pd.concat([bus_sol, fgc_sol, metro_sol, tram_sol], ignore_index=True)
all_stops = pd.read_csv('Nodes/N-All-Sol.csv')
all_stops['geometry']= all_stops['geometry'].apply(wkt.loads)
all_stops = gpd.GeoDataFrame(all_stops, geometry='geometry', crs="EPSG:4326")
len(all_stops)

215

In [30]:
(((217 * 217) *24)/2433)/3600

0.1290286340594602

In [31]:
a = alt.Chart(all_stops).mark_geoshape(size=0, color='red').encode(tooltip=['id', 'name']).properties(width=600, height=600)
neigh_layer + a

alt.LayerChart(...)

In [32]:
# cache_dir = Config().CACHE_DIR
# for f in cache_dir.glob("*.mapdb*"):
#     f.unlink()

In [33]:
transport_network = r5py.TransportNetwork(
    "Data/Neigh/cataluna-260811.osm.pbf",
    elevation_model =["Data/Neigh/elevacions-terreny-lidar-Catalunya-2m-2008-2011tif1786456535055.tif"]
    )

In [34]:
glories = shape({
    'type': 'Point',
    'coordinates':  (2.184459824628127, 41.401546206394364)
})

glories= gpd.GeoDataFrame(
    {'id' : ['Glories - City Gate'],
    'geometry': [glories]},
    crs="EPSG:4326"
)
glories

,id,geometry
0,Glories - City Gate,POINT (2.18446 41.40155)


# City Gate to all stops

In [35]:
process = 0
failed_iteraries = []
from_city_gate = r5py.DetailedItineraries(
            transport_network,
            origins=glories,
            destinations=all_stops,
            transport_modes=[r5py.TransportMode.CAR],
            snap_to_network=True,
            departure=datetime.datetime(2026, 8, 19, 13, 20),

)
from_city_gate.rename(columns={'from_id': 'origen','to_id':'dest'}, inplace=True)
from_city_gate = from_city_gate[from_city_gate['travel_time'] > pd.Timedelta(minutes=5)]
from_city_gate = from_city_gate[from_city_gate['travel_time'] < pd.Timedelta(minutes=20)]
from_city_gate = from_city_gate[['origen','dest','geometry']]
from_city_gate = from_city_gate.merge(all_stops[['id','name']], left_on='dest', right_on='id', how='left')
from_city_gate['tram'] = 'City Gate' + ' - ' + from_city_gate['name']
from_city_gate.drop(columns=['id','name'], inplace=True)
from_city_gate.to_crs('EPSG:25831', inplace=True)
from_city_gate['length'] = from_city_gate['geometry'].length
from_city_gate['speed'] = 18 /3.6 
from_city_gate['time'] = pd.to_timedelta(from_city_gate['length'] / from_city_gate['speed'],unit='s')
from_city_gate = from_city_gate[from_city_gate['time'] > pd.Timedelta(minutes=5)]
from_city_gate = from_city_gate[from_city_gate['time'] < pd.Timedelta(minutes=20)]
# from_city_gate['time'] = from_city_gate['time'].apply(
#     lambda x: f"{int(x.total_seconds() // 60):02d}:{int(x.total_seconds() % 60):02d}"
# )
from_city_gate["time"] = from_city_gate["time"].apply(
    lambda x: x.total_seconds() / 60
)
from_city_gate.to_crs('EPSG:4326', inplace=True)
from_city_gate

,origen,dest,geometry,tram,length,speed,time
2,Glories - City Gate,SB-60,"LINESTRING (2.18566 41.40098, 2.18562 41.40094...",City Gate - Llúria - Aragó,2933.664053,5.0,9.778880
3,Glories - City Gate,SB-95,"LINESTRING (2.18566 41.40098, 2.18562 41.40094...",City Gate - Gran Via - Balmes,2925.161988,5.0,9.750540
4,Glories - City Gate,SB-108,"MULTILINESTRING ((2.14619 41.37281, 2.14618 41...",City Gate - Pl Espanya - FGC,5400.529226,5.0,18.001764
6,Glories - City Gate,SB-130,"MULTILINESTRING ((2.18566 41.40098, 2.18562 41...",City Gate - Pg Lluís Companys - Parc de la Ciu...,2244.699648,5.0,7.482332
8,Glories - City Gate,SB-174,"LINESTRING (2.18566 41.40098, 2.18562 41.40094...",City Gate - Pau Claris - Aragó,2933.664053,5.0,9.778880
...,...,...,...,...,...,...,...
180,Glories - City Gate,ST-DGMR,"MULTILINESTRING ((2.18566 41.40098, 2.18562 41...",City Gate - El Maresme,4020.618102,5.0,13.402060
181,Glories - City Gate,ST-FRUM,"LINESTRING (2.18566 41.40098, 2.18562 41.40094...",City Gate - Fòrum,4217.903021,5.0,14.059677
182,Glories - City Gate,ST-STMR,"LINESTRING (2.18566 41.40098, 2.18562 41.40094...",City Gate - Sant Martí de Provençals,3681.632197,5.0,12.272107
183,Glories - City Gate,ST-BSOS,"LINESTRING (2.18566 41.40098, 2.18562 41.40094...",City Gate - Besòs,4030.277632,5.0,13.434259


In [36]:
checkpoint_dir = "route_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

stop_ids = all_stops['id'].unique()
chunk_size = 20  

for i in tqdm(range(0, len(stop_ids), chunk_size)):
    checkpoint_path = f"{checkpoint_dir}/chunk_{i:06d}.csv"
    if os.path.exists(checkpoint_path):
        continue  # already done — safe to resume after interruption

    origins = all_stops[all_stops['id'].isin(stop_ids[i:i + chunk_size])]
    chunk = r5py.DetailedItineraries(
        transport_network,
        origins=origins,
        destinations=all_stops,     
        transport_modes=[r5py.TransportMode.CAR],
        snap_to_network=True,
        departure=datetime.datetime(2026, 8, 19, 13, 20),
    )
    chunk.to_csv(checkpoint_path, index=False)

  0%|          | 0/11 [00:00<?, ?it/s]

In [37]:
across_stops = pd.concat(
    [pd.read_csv(f) for f in sorted(glob.glob(f"{checkpoint_dir}/chunk_*.csv"))],
    ignore_index=True
)
across_stops['travel_time'] = pd.to_timedelta(across_stops['travel_time'])
across_stops = across_stops[across_stops['distance'] > 0]
across_stops = across_stops[across_stops['travel_time'] > pd.Timedelta(minutes=5)]
across_stops = across_stops[across_stops['travel_time'] < pd.Timedelta(minutes=20)]

chunk it

In [38]:
final_df = pd.DataFrame()

for stop_id in tqdm(across_stops['from_id'].unique()):
    subset = across_stops[across_stops['from_id'] == stop_id]
    subset['geometry'] = subset['geometry'].apply(wkt.loads)
    final_df = pd.concat([final_df, subset], ignore_index=True)


  0%|          | 0/215 [00:00<?, ?it/s]

In [39]:
final_df = gpd.GeoDataFrame(final_df, geometry='geometry', crs="EPSG:4326")

In [40]:
final_df = final_df[['from_id', 'to_id', 'geometry']]
final_df.rename(columns={'from_id': 'origen', 'to_id': 'dest'}, inplace=True)
final_df.to_crs('EPSG:25831', inplace=True)
final_df['length'] = final_df['geometry'].length
final_df['speed'] = 18 / 3.6 
final_df['time'] = (final_df['length'] / final_df['speed']) / 60
print(len(final_df))
final_df = final_df[final_df['time'] > 5]
print(len(final_df))
final_df = final_df[final_df['time'] < 15]
print(len(final_df))
final_df.to_crs('EPSG:4326', inplace=True)
final_df = final_df.merge(all_stops[['id','name']], left_on='origen', right_on='id', how='left')
final_df.rename(columns={'name': 'origen_name'}, inplace=True)
final_df = final_df.merge(all_stops[['id','name']], left_on='dest', right_on='id', how='left')
final_df.rename(columns={'name': 'dest_name'}, inplace=True)
final_df['tram'] = final_df['origen_name'] + ' - ' + final_df['dest_name']

34150
33451
14908


In [41]:
final_df.drop(columns=['id_x','id_y','origen_name','dest_name'], inplace=True)
final_df['type'] = 'IU Trajectory'
final_df = final_df[['origen', 'dest', 'tram', 'type','length', 'speed', 'time', 'geometry']]

In [42]:
from_city_gate['type'] = 'IU Trajectory'
from_city_gate = from_city_gate[['origen', 'dest', 'tram', 'type','length', 'speed', 'time', 'geometry']]

In [43]:
all_routes = pd.concat([from_city_gate, final_df], ignore_index=True)
all_routes.insert(7, 'directed', True)


In [44]:
all_routes_layer = alt.Chart(all_routes[:1000]).mark_geoshape(
    stroke='black',
    opacity=0.5,
    filled = False
).properties(
    width=600,
    height=600
).encode(
    tooltip=['origen', 'dest', 'tram', 'type', 'length', 'speed', 'time']
)

all_stops_layer = alt.Chart(all_stops).mark_geoshape(size=0, color='red').encode(tooltip=['id', 'name']).properties(width=600, height=600)

neigh_layer + all_routes_layer + all_stops_layer

alt.LayerChart(...)

In [33]:
all_routes.to_csv('Edges/IU-Bus.csv', index=False)